In [1]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import logging
from contextlib import redirect_stderr, redirect_stdout

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracking._tracking_service.client").setLevel(logging.ERROR)

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [2]:
## Add model X, Y in this cell
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

merged_df= merged_df.drop(columns=["flag_document_5","organization_type_Industry: type 5","bureau_balance_potential_on_going_loan_loan_1","bureau_amt_credit_sum_limit_long_limit_loan_1","bureau_balance_status_score_max_loan_1","flag_document_11","own_car_age_is_missing","bureau_amt_credit_sum_limit_long_limit_loan_2"]) #,"building_score_std" #


#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

#feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "reference_for_purr.csv")
#X= clean_noise_from_feature_importance(feature_raper,X,0.0000001)
inhert_features = ['organization_type_Other industry', 'organization_type_Other trade', 'organization_type_Other', 'active_balance_months_since_delincuency_active_max', 'instalments_repeated_for_underpayment_sum_prev_1', 'instalments_potentially_on_going_prev_1', 'organization_type_Industry: type 7', 'bureau_has_bureau_balance_data_loan_1', 'organization_type_Industry: type 4', 'bureau_credit_active_active_loan_1', 'ext_source_2_is_missing', 'active_have_amt_credit_sum_overdue_active_sum', 'organization_type_Insurance', 'client_without_querys', 'bureau_credit_active_active_loan_2', 'amt_req_credit_breau_week', 'bureau_credit_active_closed_loan_1', 'instalments_extra_instalament_sum_prev_1', 'instalments_dead_tail_length_prev_1', 'flag_region_not_work', 'amt_down_payment_is_missing_mean', 'bureau_credit_active_closed_loan_2', 'bureau_credit_active_sold_loan_1', 'amt_down_payment_is_missing_sum', 'bureau_credit_active_sold_loan_2', 'bureau_credit_active_closed_loan_1', 'emergencystate_mode', 'organization_type_Industry: type 2', 'organization_type_Agriculture', 'flag_own_car', 'days_first_drawing_has_sentinel_value_prev_1', 'amt_goods_price_is_missing', 'amt_annuity_and_cnt_payment_are_missing_prev_1', 'instalments_dead_tail_length_max', 'amt_down_payment_is_missing_prev_1', 'have_sentinel_value_days_employed', 'amt_goods_price_is_missing_prev_1', 'flag_last_application_in_day_prev_1', 'rate_down_payment_is_missing_prev_1', 'rate_interesting_is_missing_prev_1', 'rate_privileged_is_missing_prev_1', 'sellerplace_area_is_missing_prev_1', 'flag_invalid_surface_sellerplace_area_prev_1', 'closed_balance_status_score_max_closed_max', 'closed_amt_credit_sum_limit_closed_min', 'organization_type_Cleaning', 'days_last_due_has_sentinel_value_prev_1', 'organization_type_Industry: type 12', 'organization_type_Postal', 'closed_amt_credit_sum_debt_closed_sum', 'organization_type_Industry: type 11', 'amt_goods_price_is_missing', 'days_and_insurance_information_are_missing_prev_1', 'organization_type_Industry: type 1', 'organization_type_Housing', 'flag_cont_mobile', 'flag_emp_phone', 'organization_type_Emergency', 'organization_type_Advertising', 'closed_credit_active_sold_closed_mean', 'closed_credit_active_sold_closed_sum', 'days_termination_has_sentinel_value_prev_1', 'organization_type_Electricity', 'bureau_balance_is_delincuency_sum_loan_1', 'organization_type_Other', 'bureau_amt_credit_sum_limit_is_missing_loan_2', 'flag_document_16', 'organization_type_Transport: type 2', 'organization_type_Transport: type 4', 'organization_type_Realtor', 'flag_document_14', 'rate_down_payment_is_missing_mean', 'rate_down_payment_is_missing_sum', 'bureau_amt_credit_sum_debt_is_negative_loan_2', 'bureau_amt_credit_sum_debt_is_negative_loan_1', 'flag_own_car', 'organization_type_XNA', 'bureau_amt_credit_sum_debt_is_negative_loan_2', 'bureau_amt_credit_sum_debt_is_negative_loan_1', 'bureau_cnt_credit_prolong_loan_2', 'bureau_amt_credit_sum_overdue_is_missing_loan_1', 'bureau_cnt_credit_prolong_loan_1', 'bureau_days_enddate_fact_is_missing_loan_2', 'bureau_days_enddate_fact_is_missing_loan_1', 'flag_document_13', 'bureau_days_credit_enddate_fourth_positive_cluster_loan_1', 'bureau_days_credit_enddate_second_positive_cluster_loan_2', 'bureau_days_credit_enddate_second_positive_cluster_loan_1', 'bureau_days_credit_enddate_first_positive_cluster_loan_2', 'bureau_days_credit_enddate_closed_loan_2', 'bureau_days_credit_enddate_is_missing_loan_2', 'bureau_days_credit_enddate_is_missing_loan_1', 'flag_document_9', 'ext_source_3_is_missing', 'bureau_credit_currency_loan_2', 'bureau_credit_currency_loan_1', 'ext_source_1_is_missing', 'bureau_amt_credit_sum_overdue_is_missing_loan_2', 'name_goods_category_prev_1', 'organization_type_Trade: type 6', 'bureau_balance_status_score_max_loan_2', 'organization_type_Trade: type 2', 'bureau_balance_months_since_delincuency_loan_2', 'info_of_social_circule_is_missing', 'amt_req_credit_berau_hour', 'organization_type_Telecom', 'organization_type_Security', 'organization_type_Restaurant', 'organization_type_Services']
#X= X.drop(columns=["flag_phone", "family_status","bureau_amt_credit_sum_limit_is_missing_loan_2", *inhert_features])
X= X.drop(columns=['closed_days_credit_closed_max'])

#X= X.drop(columns=[])

In [ ]:
columns = X.columns
indice_inicio = X.columns.get_loc("last_365_instalments_extra_instalament_mean")
columnas_restantes = X.columns[indice_inicio:]

baseline_oof_auc, baseline_std = run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"long-road-2")


results = []
features_drop_file = cfg.ARTIFACTS_DIR / 'long-road-2.csv'
pd.DataFrame(columns=['feature_dropped', 'auc_impact', 'std_impact']).to_csv(features_drop_file, index=False, encoding='utf-8')


from tqdm.auto import tqdm
for col in tqdm(columns, desc="Evaluating model without variables"):
        X_dropped = X.drop(columns=[col])
        run_name = f"long-road_2_{col.replace('/', '_')}"
        with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
            oof_auc, std = run_cv_tracked_mlflow(
                xgb.XGBClassifier, hiperparams, cv, X_dropped, Y, experiment_name, run_name=run_name
            )
        auc_drop = baseline_oof_auc - oof_auc
        std_diff = baseline_std - std
        results.append({
                    'feature_dropped': col,
                    'auc_impact': auc_drop,
                    'std_impact': std_diff
                })
        
        row_df = pd.DataFrame([{
        'feature_dropped': col,
        'auc_impact': auc_drop,
        'std_impact': std_diff
    }])
    
        row_df.to_csv(features_drop_file, mode='a', header=False, index=False, encoding='utf-8')
        print(f"Feature {col} dropped. Result: {auc_drop}, {std_diff}")



del merged_df
gc.collect()

🏃 View run long-road-test-days-closed_child_1 at: http://localhost:5332/#/experiments/3/runs/da4f65be39084182b511f4629a3c054d
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-test-days-closed_child_2 at: http://localhost:5332/#/experiments/3/runs/d4c59d9e6148418784eaf11baa659f35
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-test-days-closed_child_3 at: http://localhost:5332/#/experiments/3/runs/c72c8ca68f1c47398fa1cec41d2a57e7
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-test-days-closed_child_4 at: http://localhost:5332/#/experiments/3/runs/518d20e77b514c2b93da81ffec9abcae
🧪 View experiment at: http://localhost:5332/#/experiments/3
🏃 View run long-road-test-days-closed_child_5 at: http://localhost:5332/#/experiments/3/runs/b18b29ff2a9d45f999243183df8cc99b
🧪 View experiment at: http://localhost:5332/#/experiments/3
AUC per fold= 0.781 ± 0.002(std), auc_score_OOF= 0.781 result of CV wi

/Users/dreamcast/Documents/Home-Credit-Default-Risk-Kaggle/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run Parent_long-road-test-days-closed at: http://localhost:5332/#/experiments/3/runs/55b10eec9876436d98ea5eec6c90c0fb
🧪 View experiment at: http://localhost:5332/#/experiments/3


Evaluating model without variables:   2%|▏         | 1/57 [03:50<3:35:07, 230.50s/it]

Feature bureau_credit_type_loan_1 dropped. Result: 0.0007913627023731218, -0.0025514964609375072


...

Feature implied_interest_rate_prev_1 dropped. Result: 0.0007442111236223292, -0.0011078557412962985


Evaluating model without variables:  33%|███▎      | 19/57 [1:19:35<2:49:55, 268.30s/it]

Feature bureau_balance_months_balance_min_loan_2 dropped. Result: 0.0008794940235817839, -0.0009847825045860117


Evaluating model without variables:  33%|███▎      | 19/57 [1:19:42<2:39:25, 251.72s/it]


KeyboardInterrupt: 